In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from scipy.cluster.hierarchy import weighted

warnings.filterwarnings("ignore")


In [2]:
df =pd.read_csv(r"C:\Users\HP\Documents\EV Project\EV Station Websiite\Processed datasets\EV_usage_for_ML.csv")

In [3]:
df.head()

,vehicle_type,battery_capacity_kwh,battery_health_pct,distance_km,daily_trip_count,charging_frequency_per_week,charging_type,charging_station_distance_km,energy_consumed_kwh,electricity_cost_per_kwh,weather_condition,traffic_density,user_income_level,range_km_estimated,user_income_level.1,range_anxiety_risk,effective_battery_capacity
0,2W,3.59,83.51,37.52,1.0,4.00,public_fast,1.36,5.50,7.92,clear,0.331,middle,21.54,middle,0.0,2.998
1,2W,3.44,87.33,10.61,3.0,6.00,public_slow,1.39,1.49,8.40,humid,0.285,low,20.62,low,0.0,3.004
2,2W,3.14,98.67,34.88,3.0,4.00,home,0.17,7.06,5.70,extreme_heat,0.120,low,16.03,low,0.0,3.098
3,2W,2.06,88.74,1.00,2.0,1.00,public_fast,0.55,0.12,8.29,rainy,0.331,middle,11.10,middle,0.0,1.828
4,2W,2.64,86.47,61.73,2.0,3.33,public_fast,2.67,3.19,9.28,extreme_heat,0.068,middle,13.47,middle,1.0,2.283


In [4]:
df.drop(columns=["user_income_level.1"], inplace=True)


In [5]:
df.head()

,vehicle_type,battery_capacity_kwh,battery_health_pct,distance_km,daily_trip_count,charging_frequency_per_week,charging_type,charging_station_distance_km,energy_consumed_kwh,electricity_cost_per_kwh,weather_condition,traffic_density,user_income_level,range_km_estimated,range_anxiety_risk,effective_battery_capacity
0,2W,3.59,83.51,37.52,1.0,4.00,public_fast,1.36,5.50,7.92,clear,0.331,middle,21.54,0.0,2.998
1,2W,3.44,87.33,10.61,3.0,6.00,public_slow,1.39,1.49,8.40,humid,0.285,low,20.62,0.0,3.004
2,2W,3.14,98.67,34.88,3.0,4.00,home,0.17,7.06,5.70,extreme_heat,0.120,low,16.03,0.0,3.098
3,2W,2.06,88.74,1.00,2.0,1.00,public_fast,0.55,0.12,8.29,rainy,0.331,middle,11.10,0.0,1.828
4,2W,2.64,86.47,61.73,2.0,3.33,public_fast,2.67,3.19,9.28,extreme_heat,0.068,middle,13.47,1.0,2.283


In [6]:
df.isnull().sum()

vehicle_type                    0
battery_capacity_kwh            0
battery_health_pct              0
distance_km                     0
daily_trip_count                0
charging_frequency_per_week     0
charging_type                   0
charging_station_distance_km    0
energy_consumed_kwh             0
electricity_cost_per_kwh        0
weather_condition               0
traffic_density                 0
user_income_level               0
range_km_estimated              0
range_anxiety_risk              0
effective_battery_capacity      0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10294 entries, 0 to 10293
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   vehicle_type                  10294 non-null  str    
 1   battery_capacity_kwh          10294 non-null  float64
 2   battery_health_pct            10294 non-null  float64
 3   distance_km                   10294 non-null  float64
 4   daily_trip_count              10294 non-null  float64
 5   charging_frequency_per_week   10294 non-null  float64
 6   charging_type                 10294 non-null  str    
 7   charging_station_distance_km  10294 non-null  float64
 8   energy_consumed_kwh           10294 non-null  float64
 9   electricity_cost_per_kwh      10294 non-null  float64
 10  weather_condition             10294 non-null  str    
 11  traffic_density               10294 non-null  float64
 12  user_income_level             10294 non-null  str    
 13  range_km_est

In [9]:
df.describe()

,battery_capacity_kwh,battery_health_pct,distance_km,daily_trip_count,charging_frequency_per_week,charging_station_distance_km,energy_consumed_kwh,electricity_cost_per_kwh,traffic_density,range_km_estimated,range_anxiety_risk,effective_battery_capacity
count,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000,10294.000000
mean,12.009168,89.353359,26.339391,2.517837,3.296036,1.989392,4.337092,7.758316,0.286762,67.512976,0.239456,10.737393
std,14.494878,8.406451,16.802992,0.977802,1.452164,1.923658,2.556569,1.506419,0.157679,81.756125,0.425298,13.061034
min,1.130000,62.060000,1.000000,1.000000,1.000000,0.000000,0.110000,2.380000,0.000000,6.030000,0.000000,0.905000
25%,3.000000,83.220000,15.330000,2.000000,2.000000,0.610000,2.510000,6.690000,0.164000,16.840000,0.000000,2.652000
50%,4.605000,89.280000,25.820000,2.520000,3.000000,1.460000,4.260000,7.770000,0.271000,26.030000,0.000000,4.233000
75%,8.960000,95.470000,35.287500,3.000000,4.000000,2.690000,5.960000,8.830000,0.389000,51.330000,0.000000,8.252750
max,55.910000,116.040000,187.550000,6.000000,8.000000,17.790000,19.830000,13.090000,0.902000,325.430000,1.000000,60.331000


In [10]:
print(df.dtypes)

vehicle_type                        str
battery_capacity_kwh            float64
battery_health_pct              float64
distance_km                     float64
daily_trip_count                float64
charging_frequency_per_week     float64
charging_type                       str
charging_station_distance_km    float64
energy_consumed_kwh             float64
electricity_cost_per_kwh        float64
weather_condition                   str
traffic_density                 float64
user_income_level                   str
range_km_estimated              float64
range_anxiety_risk              float64
effective_battery_capacity      float64
dtype: object


In [11]:
## Getting all types of Features
num_feature=[feature for feature in df.columns if df[feature].dtype != 'str']
print("Num of Numerical Features:",len(num_feature))

cat_feature=[feature for feature in df.columns if df[feature].dtype == 'str']
print("Num of Categorical Features:",len(cat_feature))

discrete_feature=[feature for feature in num_feature if len(df[feature].unique())<=25]
print("Num of Discrete Features:",len(discrete_feature))

continuous_feature=[feature for feature in num_feature if feature not in discrete_feature]
print("Num of Continuous Features:",len(continuous_feature))

Num of Numerical Features: 12
Num of Categorical Features: 4
Num of Discrete Features: 3
Num of Continuous Features: 9


In [12]:
cat_feature

['vehicle_type', 'charging_type', 'weather_condition', 'user_income_level']

In [13]:
df['charging_type'].value_counts()

charging_type
public_fast    3472
public_slow    3449
home           3373
Name: count, dtype: int64

In [14]:
X=df.drop(['charging_type'], axis=1)
Y=df['charging_type']

In [15]:
X.shape

(10294, 15)

In [16]:
Y.shape

(10294,)

In [17]:
df['vehicle_type'].value_counts()



vehicle_type
2W    5152
3W    3045
4W    2097
Name: count, dtype: int64

In [18]:
## Label Encoding for charging type feature
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
Y=le.fit_transform(Y)

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [20]:
## OneHot encoding for categorical features
numeric_feature=X.select_dtypes(exclude='str').columns
onehot_columns=['vehicle_type', 'weather_condition', 'user_income_level']

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
OneHot_Transformer=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",OneHot_Transformer,onehot_columns),
        ("StandardScaler",numeric_transformer,numeric_feature),
    ]
)


In [21]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [22]:
pd.DataFrame(X)

,vehicle_type,battery_capacity_kwh,battery_health_pct,distance_km,daily_trip_count,charging_frequency_per_week,charging_station_distance_km,energy_consumed_kwh,electricity_cost_per_kwh,weather_condition,traffic_density,user_income_level,range_km_estimated,range_anxiety_risk,effective_battery_capacity
0,2W,3.59,83.51,37.52,1.00,4.00,1.36,5.50,7.92,clear,0.331,middle,21.54,0.0,2.998
1,2W,3.44,87.33,10.61,3.00,6.00,1.39,1.49,8.40,humid,0.285,low,20.62,0.0,3.004
2,2W,3.14,98.67,34.88,3.00,4.00,0.17,7.06,5.70,extreme_heat,0.120,low,16.03,0.0,3.098
3,2W,2.06,88.74,1.00,2.00,1.00,0.55,0.12,8.29,rainy,0.331,middle,11.10,0.0,1.828
4,2W,2.64,86.47,61.73,2.00,3.33,2.67,3.19,9.28,extreme_heat,0.068,middle,13.47,1.0,2.283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10289,4W,34.27,89.47,43.03,4.00,6.00,2.49,7.28,8.15,rainy,0.230,middle,185.04,0.0,30.661
10290,4W,34.13,99.95,59.65,2.00,2.00,1.20,8.10,6.69,humid,0.166,middle,204.80,0.0,34.113
10291,4W,33.52,83.97,61.25,2.53,4.00,0.75,12.10,9.67,humid,0.525,middle,201.09,0.0,28.147
10292,4W,31.09,90.34,28.43,2.00,3.00,1.52,4.63,8.72,rainy,0.153,high,167.89,0.0,28.087


In [24]:
X_test

array([[ 0.        ,  0.        ,  1.        , ...,  2.75077625,
        -0.56214625,  2.63490496],
       [ 1.        ,  0.        ,  0.        , ..., -0.64060638,
        -0.56214625, -0.63445128],
       [ 0.        ,  1.        ,  0.        , ..., -0.31287539,
        -0.56214625, -0.31638966],
       ...,
       [ 1.        ,  0.        ,  0.        , ..., -0.62806689,
        -0.56214625, -0.62217709],
       [ 0.        ,  1.        ,  0.        , ..., -0.22278589,
        -0.56214625, -0.23176637],
       [ 1.        ,  0.        ,  0.        , ..., -0.66349398,
        -0.56214625, -0.63269782]], shape=(2059, 22))

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

In [30]:
models={
    "Random Forest Classifier": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier()
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,Y_train)

    ##Make Prediction
    Y_train_pred=model.predict(X_train)
    Y_test_pred=model.predict(X_test)

    ## Training data performance
    model_train_accuracy=accuracy_score(Y_train,Y_train_pred)
    model_train_precision=precision_score(Y_train,Y_train_pred)
    model_train_recall=recall_score(Y_train,Y_train_pred)
    model_train_f1=f1_score(Y_train,Y_train_pred,average="weighted")

    ##Testing data performance
    model_test_accuracy=accuracy_score(Y_test,Y_test_pred)
    model_test_precision=precision_score(Y_test,Y_test_pred,average="binary")
    model_test_recall=recall_score(Y_test,Y_test_pred,average="binary")
    model_test_f1=f1_score(Y_test,Y_test_pred,average="weighted")

    print(list(models)[i])

    print("Performance of the Training Data")
    print("Accuracy: {:.4f}".format(model_train_accuracy))
    print("Precision: {:.4f}".format(model_train_precision))
    print("Recall: {:.4f}".format(model_train_recall))
    print("F1_score: {:.4f}".format(model_train_f1))

    print("Performance of the Testing Data")
    print("Accuracy: {:.4f}".format(model_test_accuracy))
    print("Precision: {:.4f}".format(model_test_precision))
    print("Recall: {:.4f}".format(model_test_recall))
    print("F1_score: {:.4f}".format(model_test_f1))

    print('='*35)
    print('\n')

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].